# 3. Data Cleaning

## 1. Introduction

Data cleaning is the process of identifying and handling
data quality issues.

The main objectives are:

- Validate the dataset structure.
- Verify the target variable.
- Identify infinite values.
- Investigate duplicate records.
- Check missing values.
- Prepare a clean dataset for further analysis.

Data cleaning must be performed carefully to avoid removing
valid financial transactions.

In [1]:
import numpy as np
import pandas as pd

from IPython.display import display

from src.data_loader import load_data
from src.validation import (
    validate_columns,
    validate_target,
    validate_numeric_columns,
)

from src.cleaning import (
    replace_infinite_values,
    remove_exact_duplicates,
    clean_data,
    save_cleaned_data,
)

from src.config import RAW_DATA_PATH, TARGET_COLUMN

df = load_data(RAW_DATA_PATH)

print("Original Dataset Shape:", df.shape)

display(df.head())

Original Dataset Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Initial Data Validation

Before modifying the dataset, the required columns and target
variable must be validated.

The target variable must contain only binary values:

- 0: Legitimate
- 1: Fraudulent

The feature columns are expected to be numerical.

In [2]:
validate_columns(df)

validate_target(df)

validate_numeric_columns(df)

print("Column validation passed.")
print("Target validation passed.")
print("Numeric validation passed.")

Column validation passed.
Target validation passed.
Numeric validation passed.


## 3. Handling Infinite Values

Infinite values cannot be processed reliably by many
Machine Learning algorithms.

Positive and negative infinity values will be replaced
with missing values.

The missing values will be handled later inside the
preprocessing pipeline.

In [3]:
df_clean = replace_infinite_values(df)

print("Infinite values before cleaning:")
print(np.isinf(df.select_dtypes(include=np.number)).sum().sum())

print("\nMissing values after replacing infinity:")
print(df_clean.isnull().sum().sum())

Infinite values before cleaning:
0

Missing values after replacing infinity:
0


## 4. Duplicate Records Investigation

Exact duplicate records are identified.

Duplicate removal is not automatically applied because
repeated transaction records may require domain investigation.

For this project, duplicate removal will remain disabled
unless the records are confirmed to be invalid duplicates.

In [4]:
duplicate_count = df_clean.duplicated().sum()

print("Exact Duplicate Rows:", duplicate_count)

duplicate_rows = df_clean[
    df_clean.duplicated(keep=False)
]

display(duplicate_rows.head(10))

Exact Duplicate Rows: 1081


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
32,26.0,-0.529912,0.873892,1.347247,0.145457,0.414209,0.100223,0.711206,0.176066,-0.286717,...,0.046949,0.208105,-0.185548,0.001031,0.098816,-0.552904,-0.073288,0.023307,6.14,0
33,26.0,-0.529912,0.873892,1.347247,0.145457,0.414209,0.100223,0.711206,0.176066,-0.286717,...,0.046949,0.208105,-0.185548,0.001031,0.098816,-0.552904,-0.073288,0.023307,6.14,0
34,26.0,-0.535388,0.865268,1.351076,0.147575,0.433680,0.086983,0.693039,0.179742,-0.285642,...,0.049526,0.206537,-0.187108,0.000753,0.098117,-0.553471,-0.078306,0.025427,1.77,0
35,26.0,-0.535388,0.865268,1.351076,0.147575,0.433680,0.086983,0.693039,0.179742,-0.285642,...,0.049526,0.206537,-0.187108,0.000753,0.098117,-0.553471,-0.078306,0.025427,1.77,0
112,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
113,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
114,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
115,74.0,1.038370,0.127486,0.184456,1.109950,0.441699,0.945283,-0.036715,0.350995,0.118950,...,0.102520,0.605089,0.023092,-0.626463,0.479120,-0.166937,0.081247,0.001192,1.18,0
220,145.0,-2.420413,1.947885,0.553646,0.983069,-0.281518,2.408958,-1.401613,-0.188299,0.675878,...,1.213826,-1.238620,0.006927,-1.724222,0.239603,-0.313703,-0.188281,0.119831,6.00,0
221,145.0,-2.420413,1.947885,0.553646,0.983069,-0.281518,2.408958,-1.401613,-0.188299,0.675878,...,1.213826,-1.238620,0.006927,-1.724222,0.239603,-0.313703,-0.188281,0.119831,6.00,0


## 5. Cleaning Strategy

The following cleaning decisions are applied:

1. Replace infinite values with NaN.
2. Preserve duplicate records by default.
3. Do not remove rows because of missing feature values.
4. Never impute missing target values.
5. Perform imputation later using training data only.

This approach minimizes the risk of information leakage.

In [5]:
df_clean = clean_data(
    df,
    remove_duplicates=False
)

print("Cleaning completed.")

print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)

display(df_clean.head())

Cleaning completed.
Original Shape: (284807, 31)
Cleaned Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [6]:
quality_report = pd.DataFrame({
    "Data Type": df_clean.dtypes,
    "Missing Values": df_clean.isnull().sum(),
    "Missing Percentage": (
        df_clean.isnull().mean() * 100
    ),
    "Unique Values": df_clean.nunique()
})

display(quality_report)

,Data Type,Missing Values,Missing Percentage,Unique Values
Time,float64,0,0.0,124592
V1,float64,0,0.0,275663
V2,float64,0,0.0,275663
V3,float64,0,0.0,275663
V4,float64,0,0.0,275663
V5,float64,0,0.0,275663
V6,float64,0,0.0,275663
V7,float64,0,0.0,275663
V8,float64,0,0.0,275663
V9,float64,0,0.0,275663


## 6. Saving the Cleaned Dataset

The cleaned dataset is saved in the processed data directory.

The original raw dataset remains unchanged.

The processed dataset will be used in the subsequent
exploratory data analysis and modeling stages.

In [7]:
save_cleaned_data(df_clean)

print("Cleaned dataset saved successfully.")

Cleaned data saved to: D:\Python-Workspace\Projects\Data-Science\Credit-Card-Fraud-Detection\data\processed\cleaned_data.csv
Cleaned dataset saved successfully.


In [8]:
from src.config import CLEANED_DATA_PATH

df_saved = pd.read_csv(CLEANED_DATA_PATH)

print("Saved Dataset Shape:", df_saved.shape)

assert df_saved.shape == df_clean.shape

print("Saved dataset verification passed.")

Saved Dataset Shape: (284807, 31)
Saved dataset verification passed.
